In [ ]:
%python
dbutils.widgets.text("config_id",          "", "Ingestion config ID")
dbutils.widgets.text("landing_path",       "", "S3 landing path to read from")
dbutils.widgets.text("file_format",        "parquet", "Format landing_path was written in")
dbutils.widgets.text("silver_catalog",     "", "Silver target catalog")
dbutils.widgets.text("silver_schema",      "", "Silver target schema (Bronze's target_schema + '_silver')")
dbutils.widgets.text("silver_table",       "", "Silver target table (same name as Bronze's target_table)")
dbutils.widgets.text("source_schema",      "", "Source schema")
dbutils.widgets.text("source_object_name", "", "Source object/table name")
dbutils.widgets.text("load_type",          "", "Load type")
dbutils.widgets.text("primary_key_cols",   "", "Comma-separated primary key columns")

config_id           = dbutils.widgets.get("config_id")
landing_path        = dbutils.widgets.get("landing_path")
file_format         = dbutils.widgets.get("file_format") or "parquet"
silver_catalog      = dbutils.widgets.get("silver_catalog")
silver_schema       = dbutils.widgets.get("silver_schema")
silver_table        = dbutils.widgets.get("silver_table")
source_schema       = dbutils.widgets.get("source_schema")
source_object_name  = dbutils.widgets.get("source_object_name")
load_type           = dbutils.widgets.get("load_type")
primary_key_cols    = dbutils.widgets.get("primary_key_cols")

silver_full_table = f"{silver_catalog}.{silver_schema}.{silver_table}"
print(f"✅ Silver running for config_id={config_id} landing_path={landing_path} → {silver_full_table}")

In [ ]:
%python
from pyspark.sql import functions as F

df = spark.read.format(file_format).load(landing_path)

# TODO: real Bronze → Silver transformation (dedupe on primary_key_cols, type
# casting, column renames, etc.) goes here. Currently a straight pass-through.
df_out = df.withColumn("_silver_written_at", F.current_timestamp())

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")
df_out.writeTo(silver_full_table).using("delta").createOrReplace()

rows_written = df_out.count()
print(f"✅ Silver SUCCESS — wrote {rows_written} rows to {silver_full_table}")